# RAG Dataset Preparation

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("osamahosamabdellatif/high-quality-invoice-images-for-ocr")

print("Path to dataset files:", path)

/home/uplong/Documents/makeathon_2026/makeathon-2026-WHOAMI-inform/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/uplong/.cache/kagglehub/datasets/osamahosamabdellatif/high-quality-invoice-images-for-ocr/versions/3


In [2]:
import pandas as pd


df = pd.read_csv(path+'/batch_1/batch_1/batch1_1.csv')    #/home/uplong/.cache/kagglehub/datasets/osamahosamabdellatif/high-quality-invoice-images-for-ocr/versions/3/batch_1/batch_1

sample = df.iloc[100]['Json Data']

In [3]:
from datetime import datetime
from decimal import Decimal, InvalidOperation
import json
import re


def normalize_invoice(data: dict) -> dict:
    """
    Mapping of the json formats
    """    
    invoice = data.get("invoice", {}) or {}
    items = data.get("items", []) or []
    subtotal_data = data.get("subtotal", {}) or {}
    payment = data.get("payment_instructions", {}) or {}

    def clean_string(value):
        if value is None:
            return None
        value = str(value).strip()
        return value if value else None

    def to_decimal(value):
        value = clean_string(value)
        if value is None:
            return None
        try:
            cleaned = value.replace(",", "")
            return Decimal(cleaned)
        except (InvalidOperation, ValueError):
            return None

    def to_float(value):
        dec = to_decimal(value)
        return float(dec) if dec is not None else None

    def to_iso_date(value):
        value = clean_string(value)
        if not value:
            return None

        for fmt in ("%m/%d/%Y", "%Y-%m-%d", "%m-%d-%Y"):
            try:
                return datetime.strptime(value, fmt).date().isoformat()
            except ValueError:
                pass

        try:
            return datetime.fromisoformat(value).date().isoformat()
        except ValueError:
            return None

    # def normalize_vendor_name(name):
    #     name = clean_string(name)
    #     if not name:
    #         return None
    #     normalized = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    #     return normalized or None

    def normalize_vendor_name(name):
        name = clean_string(name)
        if not name:
            return None
        return name.lower()  # matches what's in ChromaDB: "nguyen-roach", "bailey, murray and lewis"

    vendor = clean_string(invoice.get("seller_name"))
    invoice_number = clean_string(invoice.get("invoice_number"))
    invoice_date = to_iso_date(invoice.get("invoice_date"))
    due_date = to_iso_date(invoice.get("due_date") or payment.get("due_date"))

    # ✅ Compute financials FIRST
    tax = to_float(subtotal_data.get("tax"))
    discount = to_float(subtotal_data.get("discount")) or 0.0
    total = to_float(subtotal_data.get("total"))

    computed_subtotal = None
    if total is not None:
        computed_subtotal = round(total - (tax or 0.0) + discount, 2)

    # ✅ Now build raw_text — total is defined
    item_descs = " | ".join(
        item.get("description", "").replace("\\n", " ").strip()
        for item in items
        if item.get("description")
    )

    d = (
        f"Invoice {invoice_number} from {vendor} dated {invoice_date}. "
        f"Items: {item_descs}. "
        f"Total: {total} USD."
    )

    # ✅ Then build line_items
    line_items = []
    for item in items:
        qty = to_float(item.get("quantity"))
        line_total = to_float(item.get("total_price"))
        unit_price = round(line_total / qty, 2) if qty and line_total is not None else None
        line_items.append({
            "description": clean_string(item.get("description")),
            "qty": qty,
            "unit_price": unit_price,
            "line_total": line_total
        })

    result = {
        "doc_id": f"inv_{invoice_number}" if invoice_number else "inv_unknown",
        "doc_type": "invoice",
        "vendor": vendor,
        "vendor_normalized": normalize_vendor_name(vendor),
        "invoice_number": invoice_number,
        "invoice_date": invoice_date,
        "due_date": due_date,
        "currency": "USD",
        "subtotal": computed_subtotal,
        "tax": tax if tax is not None else 0.0,
        "total": total,
        "status": "open" if due_date else "unknown",
        "line_items": line_items,
        "raw_text": d
    }

    return result

In [4]:
test = df['Json Data'].apply(lambda x: normalize_invoice(json.loads(x.strip())))
test

0      {'doc_id': 'inv_84652373', 'doc_type': 'invoic...
1      {'doc_id': 'inv_37451664', 'doc_type': 'invoic...
2      {'doc_id': 'inv_40108666', 'doc_type': 'invoic...
3      {'doc_id': 'inv_73285932', 'doc_type': 'invoic...
4      {'doc_id': 'inv_15288019', 'doc_type': 'invoic...
                             ...                        
494    {'doc_id': 'inv_87873157', 'doc_type': 'invoic...
495    {'doc_id': 'inv_29534383', 'doc_type': 'invoic...
496    {'doc_id': 'inv_77715322', 'doc_type': 'invoic...
497    {'doc_id': 'inv_35163496', 'doc_type': 'invoic...
498    {'doc_id': 'inv_68766191', 'doc_type': 'invoic...
Name: Json Data, Length: 499, dtype: object

In [5]:
import pandas as pd


df = pd.read_csv(path+'/batch_1/batch_1/batch1_1.csv')    #/home/uplong/.cache/kagglehub/datasets/osamahosamabdellatif/high-quality-invoice-images-for-ocr/versions/3/batch_1/batch_1

sample = df.iloc[100]['Json Data']

# print(sample.strip())

In [6]:
import pandas as pd
import json

import pandas as pd
import json

def series_json_to_df(s: pd.Series) -> pd.DataFrame:
    def parse(x):
        if pd.isna(x):
            return {}
        if isinstance(x, dict):
            return x
        if isinstance(x, str):
            return json.loads(x)
        return x

    return pd.json_normalize(s.map(parse), max_level=0)


new_df = series_json_to_df(test)

new_df['filename'] = df['File Name']

new_df = new_df.loc[:, ['filename', 'doc_id',
 'doc_type',
 'vendor',
 'vendor_normalized',
 'invoice_number',
 'invoice_date',
 'due_date',
 'currency',
 'subtotal',
 'tax',
 'total',
 'status',
 'line_items',
 'raw_text',
 ]]

new_df['line_items'] = new_df['line_items'].astype(str)


In [7]:
new_df.columns.to_list()

['filename',
 'doc_id',
 'doc_type',
 'vendor',
 'vendor_normalized',
 'invoice_number',
 'invoice_date',
 'due_date',
 'currency',
 'subtotal',
 'tax',
 'total',
 'status',
 'line_items',
 'raw_text']

In [8]:
new_df['vendor_normalized'] = new_df['vendor'].apply(lambda x: x.lower())

new_df.head()

,filename,doc_id,doc_type,vendor,vendor_normalized,invoice_number,invoice_date,due_date,currency,subtotal,tax,total,status,line_items,raw_text
0,batch1-0494.jpg,inv_84652373,invoice,Nguyen-Roach,nguyen-roach,84652373,2021-02-23,None,USD,211.77,21.18,232.95,unknown,[{'description': 'Stemware Rack Display Kitche...,Invoice 84652373 from Nguyen-Roach dated 2021-...
1,batch1-0489.jpg,inv_37451664,invoice,Scott-Howard,scott-howard,37451664,2020-06-11,None,USD,139.93,13.99,153.92,unknown,[{'description': 'PUMA Boys Youth Universal FG...,Invoice 37451664 from Scott-Howard dated 2020-...
2,batch1-0499.jpg,inv_40108666,invoice,"Bailey, Murray and Lewis","bailey, murray and lewis",40108666,2020-02-07,None,USD,452.92,45.29,498.21,unknown,[{'description': 'Microsoft Xbox One 1 - 500 G...,"Invoice 40108666 from Bailey, Murray and Lewis..."
3,batch1-0497.jpg,inv_73285932,invoice,"Merritt, Williams and Young","merritt, williams and young",73285932,2017-07-25,None,USD,615.88,61.59,677.47,unknown,[{'description': 'Bcbgeneration Black Sleevele...,"Invoice 73285932 from Merritt, Williams and Yo..."
4,batch1-0081.jpg,inv_15288019,invoice,Fernandez Ltd,fernandez ltd,15288019,2014-09-07,None,USD,1.55,0.16,1.71,unknown,[{'description': 'Easy No Tie Rubber Shoe Lace...,Invoice 15288019 from Fernandez Ltd dated 2014...


# Make VectorDB

In [9]:
import pandas as pd
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from typing import List

In [10]:

# ============================================================
# END-TO-END: DataFrame → ChromaDB
# Model: FinLang/finance-embeddings-investopedia (768-dim)
# 2 separate chunks per row: 'lineitems' and 'rawtext'
# ============================================================
# pip install chromadb sentence-transformers pandas tqdm
# ============================================================

import pandas as pd
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from typing import List

# -----------------------------------------------------------
# 1. Custom embedding function using FinLang finance model
# -----------------------------------------------------------
class FinLangEmbeddingFunction(EmbeddingFunction):
    def __init__(self, model_name: str = "FinLang/finance-embeddings-investopedia", device: str = "cpu"):
        print(f"Loading model '{model_name}'... (first run downloads ~440MB)")
        self.model = SentenceTransformer(model_name, device=device)

    def __call__(self, input: Documents) -> Embeddings:
        embeddings = self.model.encode(
            list(input),
            normalize_embeddings=True,  # cosine similarity ready
            show_progress_bar=False,
        )
        return embeddings.tolist()

# -----------------------------------------------------------
# 2. Load the dataframe
# -----------------------------------------------------------

col1, col2 = new_df.columns[-2], new_df.columns[-1]   # 'lineitems', 'rawtext'
print(f"Chunking columns : '{col1}'  and  '{col2}'")
print(f"Total rows       : {len(new_df)}")

# -----------------------------------------------------------
# 3. Build 2 chunk records per row
# -----------------------------------------------------------
records = []

for idx, row in new_df.iterrows():
    inv_date_str = row.get("invoice_date", "")
    inv_date_int = None
    if pd.notna(inv_date_str) and str(inv_date_str).strip():
        try:
            inv_date_int = int(str(inv_date_str).replace("-", ""))
        except ValueError:
            pass

    meta_base = {
        "filename":          str(row.get("filename",           "")),
        "doc_id":            str(row.get("doc_id",             "")),
        "doc_type":          str(row.get("doc_type",           "")),
        "vendor":            str(row.get("vendor",             "")),
        "vendor_normalized": str(row.get("vendor_normalized",  "")),
        "invoice_number":    str(row.get("invoice_number",     "")),
        "invoice_date":      str(row.get("invoice_date",       "")),
        "invoice_date_int":  inv_date_int,
        "currency":          str(row.get("currency",           "")),
        "subtotal": float(row["subtotal"]) if pd.notna(row.get("subtotal")) else 0.0,
        "tax":      float(row["tax"])      if pd.notna(row.get("tax"))      else 0.0,
        "total":    float(row["total"])    if pd.notna(row.get("total"))    else 0.0,
        "status":            str(row.get("status",             "")),
    }
    
    

    for field in [col1, col2]:
        text = str(row[field]).strip() if pd.notna(row[field]) else ""
        if not text or text.lower() in ("nan", "none", ""):
            continue  # skip empty chunks

        records.append({
            "id":       f"{idx}_{field}",
            "document": text,
            "metadata": {**meta_base, "source_column": field},
        })

print(f"Total chunks     : {len(records)}")



Chunking columns : 'line_items'  and  'raw_text'
Total rows       : 499
Total chunks     : 998


In [11]:
records[0]

{'id': '0_line_items',
 'document': '[{\'description\': \'Stemware Rack Display Kitchen\\nWine Glass Holder Bottle\\nCarbon Steel Free Punch\', \'qty\': 1.0, \'unit_price\': 46.55, \'line_total\': 46.55}, {\'description\': \'VTG (4) 7 Ounce Since 1852\\nMilk Bottle Wine Carafe Juice\\nGlass with Cork Lids\', \'qty\': 1.0, \'unit_price\': 15.4, \'line_total\': 15.4}, {\'description\': \'Vintage Crystal Red Wine\\nGlasses NOS West Germany\\n1983 6 10 ounce elegant stems\', \'qty\': 1.0, \'unit_price\': 39.0, \'line_total\': 39.0}, {\'description\': \'3 Ikea Stainless Steel 4-bottle\\nWine Rack 300.557.60 - great\\ncondition gift it!\', \'qty\': 4.0, \'unit_price\': 27.5, \'line_total\': 110.0}, {\'description\': \'Lolita "Wine Bouquet" Hand\\nPainted and Decorated Wine\\nGlass NIB\', \'qty\': 1.0, \'unit_price\': 22.0, \'line_total\': 22.0}]',
 'metadata': {'filename': 'batch1-0494.jpg',
  'doc_id': 'inv_84652373',
  'doc_type': 'invoice',
  'vendor': 'Nguyen-Roach',
  'vendor_normalized

In [14]:
# -----------------------------------------------------------
# 4. Init ChromaDB with FinLang embedding function
# -----------------------------------------------------------
embedding_fn = FinLangEmbeddingFunction(
    model_name="FinLang/finance-embeddings-investopedia",
    device="cpu",   # change to "cuda" if you have a GPU
)

client = chromadb.PersistentClient(path="./invoices_chroma_db")

# Delete existing collection if re-running from scratch (optional)
try:
    client.delete_collection("invoices")
    print("Deleted existing 'invoices' collection.")
except Exception:
    pass

collection = client.get_or_create_collection(
    name="invoices", # tabla name
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},  # matches normalize_embeddings=True
)

# -----------------------------------------------------------
# 5. Batch upsert — tune BATCH_SIZE based on your RAM
# -----------------------------------------------------------
BATCH_SIZE = 64

for i in tqdm(range(0, len(records), BATCH_SIZE), desc="Inserting batches"):
    batch = records[i : i + BATCH_SIZE]
    collection.upsert(
        ids=       [r["id"]       for r in batch],
        documents= [r["document"] for r in batch],
        metadatas= [r["metadata"] for r in batch],
    )

print(f"\n✅ Done! '{collection.name}' contains {collection.count()} vectors.")

# -----------------------------------------------------------
# 6. Sanity-check query
# -----------------------------------------------------------
print("\n--- Query: 'invoice for gaming computers' ---")
results = collection.query(
    query_texts=["invoice for gaming computers"],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
)):
    print(f"\nResult #{i+1}  (cosine distance: {dist:.4f})")
    print(f"  Source column : {meta['source_column']}")
    print(f"  Vendor        : {meta['vendor']}")
    print(f"  Invoice #     : {meta['invoice_number']}")
    print(f"  Text snippet  : {doc[:200]}...")

# -----------------------------------------------------------
# 7. Filter query — search only in 'lineitems' chunks
# -----------------------------------------------------------
print("\n--- Filter query: 'Nike shoes' in lineitems only ---")
results_filtered = collection.query(
    query_texts=["Nike shoes"],
    n_results=3,
    where={"source_column": col1},   # only lineitems chunks
    include=["documents", "metadatas", "distances"],
)

for i, (doc, meta, dist) in enumerate(zip(
    results_filtered["documents"][0],
    results_filtered["metadatas"][0],
    results_filtered["distances"][0],
)):
    print(f"\nResult #{i+1}  (cosine distance: {dist:.4f})")
    print(f"  Vendor       : {meta['vendor']}")
    print(f"  Text snippet : {doc[:200]}...")

Loading model 'FinLang/finance-embeddings-investopedia'... (first run downloads ~440MB)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14785.16it/s]


Deleted existing 'invoices' collection.


Inserting batches: 100%|██████████| 16/16 [02:38<00:00,  9.89s/it]


✅ Done! 'invoices' contains 998 vectors.

--- Query: 'invoice for gaming computers' ---

Result #1  (cosine distance: 0.3956)
  Source column : raw_text
  Vendor        : Collins Group
  Invoice #     : 86225018
  Text snippet  : Invoice 86225018 from Collins Group dated 2015-12-18. Items: Fast i5 RGB Gaming Desktop PC Computer nVidia Gefore WiFi Fortnite GTAV PUBG IG3 | FAST Dell Optiplex Windows 10 Desktop Computer Tower C2D...

Result #2  (cosine distance: 0.3970)
  Source column : raw_text
  Vendor        : Atkinson-Woods
  Invoice #     : 11640046
  Text snippet  : Invoice 11640046 from Atkinson-Woods dated 2014-11-06. Items: Dell Desktop Computer Intel Core i5 Windows 10 pro 64 250gb 3.1ghz 4gb Ram | Custom Gaming Desktop PC Computer - Core i5 4690, RX 470, 8GB...

Result #3  (cosine distance: 0.4059)
  Source column : raw_text
  Vendor        : Mendoza-Gonzalez
  Invoice #     : 68350338
  Text snippet  : Invoice 68350338 from Mendoza-Gonzalez dated 2012-04-25. Items: Dell Opti

In [16]:
sample = collection.get(limit=5, include=["metadatas"])
for m in sample["metadatas"]:
    print(m.get("vendor"), "→", m.get("vendor_normalized"))

Nguyen-Roach → nguyen-roach
Nguyen-Roach → nguyen-roach
Scott-Howard → scott-howard
Scott-Howard → scott-howard
Bailey, Murray and Lewis → bailey, murray and lewis


# Prefiltering

In [17]:
# MAKE THE VENDOR MAPPING


# Run once after new_df is ready, before or after ingestion
vendor_lookup = {
    row["vendor_normalized"]: row["vendor_normalized"]
    for _, row in new_df.iterrows()
    if pd.notna(row.get("vendor_normalized")) and row.get("vendor_normalized")
}

# Also map the human-readable vendor name → normalized form
# so "Nguyen-Roach" in a query also matches "nguyen_roach" in the DB
for _, row in new_df.iterrows():
    vendor = str(row.get("vendor", "")).strip().lower()
    normalized = row.get("vendor_normalized", "")
    if vendor and normalized:
        vendor_lookup[vendor] = normalized

print(f"Built vendor lookup with {len(vendor_lookup)} entries.")

Built vendor lookup with 493 entries.


In [18]:
from __future__ import annotations

import re
import json
from datetime import datetime
from typing import Any, Optional
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
from sentence_transformers import SentenceTransformer


# =========================
# 1. EMBEDDING FUNCTION
# =========================

class FinLangEmbeddingFunction(EmbeddingFunction):
    def __init__(self, model_name: str = "FinLang/finance-embeddings-investopedia", device: str = "cpu"):
        print(f"Loading model '{model_name}'...")
        self.model = SentenceTransformer(model_name, device=device)

    def __call__(self, input: Documents) -> Embeddings:
        return self.model.encode(
            list(input),
            normalize_embeddings=True,
            show_progress_bar=False,
        ).tolist()


# =========================
# 2. HELPERS
# =========================

MONTHS = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12
}

STOPWORDS = {
    "show", "me", "find", "the", "a", "an", "from", "for", "of", "in",
    "all", "with", "documents", "document", "please", "get", "list",
    "give", "search", "look", "fetch"
}


def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-zA-Z0-9\.]+", text.lower())


def month_range(year: int, month: int) -> tuple[str, str]:
    from datetime import datetime
    start = datetime(year, month, 1)
    end_month = month + 1 if month < 12 else 1
    end_year = year if month < 12 else year + 1
    end = datetime(end_year, end_month, 1)
    return start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")


# =========================
# 3. QUERY PARSER
# =========================
# This is rule-based but mirrors what a cheap LLM parser would return.
# Replace parse_user_query() with an LLM call returning the same dict
# to make this production-ready.
#
# Expected output schema:
# {
#   "intent": str,
#   "semantic_query": str,
#   "chroma_where": dict | None,   <-- native ChromaDB $and/$eq/$gte filter
#   "source_column": str | None,   <-- "line_items" or "rawtext" or None (both)
#   "amount_gt": float | None,     <-- NOTE: stored as str in chroma, handle carefully
#   "amount_lt": float | None,
# }

def parse_user_query(query: str, vendor_lookup: dict) -> dict[str, Any]:
    q = query.lower()

    filters: list[dict] = []
    source_column: Optional[str] = None
    amount_gt: Optional[float] = None
    amount_lt: Optional[float] = None

    # ----- doc type -----
    if "invoice" in q:
        filters.append({"doc_type": {"$eq": "invoice"}})
    elif "receipt" in q:
        filters.append({"doc_type": {"$eq": "receipt"}})

    # ----- source column -----
    if "line item" in q or "line_item" in q or "lineitems" in q or "items" in q:
        source_column = "line_items"
        filters.append({"source_column": {"$eq": "line_items"}})
    elif "raw text" in q or "rawtext" in q or "description" in q:
        source_column = "rawtext"
        filters.append({"source_column": {"$eq": "rawtext"}})

    # ----- vendor (exact known names → $eq; fuzzy left to semantic search) -----
    # You can expand this dict with all known vendor names from your dataset
    # known_vendors = {
    #     "adobe": "adobe",
    #     "figma": "figma",
    #     "nguyen": "nguyen-roach",
    #     "nguyen-roach": "nguyen-roach",
    # }
    # for keyword, normalized in vendor_lookup.items():
    #     if keyword in q:
    #         filters.append({"vendor_normalized": {"$eq": normalized}})
    #         break
    
    sorted_vendor_keys = sorted(vendor_lookup.keys(), key=len, reverse=True)

    matched_vendor = None
    for keyword in sorted_vendor_keys:
        if keyword in q:
            matched_vendor = vendor_lookup[keyword]
            break

    if matched_vendor:
        filters.append({"vendor_normalized": {"$eq": matched_vendor}})

    # ----- invoice number -----
    inv_match = re.search(r"\b\d{6,10}\b", q)
    if inv_match:
        filters.append({"invoice_number": {"$eq": inv_match.group(0)}})

    # ----- currency -----
    if "$" in q or "usd" in q or "dollar" in q:
        filters.append({"currency": {"$eq": "USD"}})
    elif "€" in q or "eur" in q or "euro" in q:
        filters.append({"currency": {"$eq": "EUR"}})

    # ----- status -----
    if "paid" in q:
        filters.append({"status": {"$eq": "paid"}})
    elif "unpaid" in q or "outstanding" in q or "overdue" in q:
        filters.append({"status": {"$in": ["unpaid", "outstanding", "overdue"]}})

    # ----- date: month + year -----
    month_match = re.search(
        r"\b(january|february|march|april|may|june|july|august|september|october|november|december)\s+(\d{4})\b",
        q
    )
    if month_match:
        month_name = month_match.group(1)
        year = int(month_match.group(2))
        month = MONTHS[month_name]
        date_start, date_end = month_range(year, month)
        # ChromaDB requires numeric values for /
        date_start_int = int(date_start.replace("-", ""))
        date_end_int = int(date_end.replace("-", ""))
        filters.append({"invoice_date_int": {"$gte": date_start_int}})
        filters.append({"invoice_date_int": {"$lt": date_end_int}})
    # ----- year only -----
    elif re.search(r"\bin\s+(\d{4})\b", q):
        year_match = re.search(r"\bin\s+(\d{4})\b", q)
        year = int(year_match.group(1))
        filters.append({"invoice_date_int": {"$gte": int(f"{year}0101")}})
        filters.append({"invoice_date_int": {"$lt": int(f"{year + 1}0101")}})

    # ----- amount (stored as string in ChromaDB, so we track separately) -----
    gt_match = re.search(r"(?:over|greater than|above|more than)\s*[$€]?\s*(\d+(?:\.\d+)?)", q)
    lt_match = re.search(r"(?:under|less than|below)\s*[$€]?\s*(\d+(?:\.\d+)?)", q)
    if gt_match:
        amount_gt = float(gt_match.group(1))
    if lt_match:
        amount_lt = float(lt_match.group(1))

    # ----- build ChromaDB where clause -----
    if len(filters) == 0:
        chroma_where = None
    elif len(filters) == 1:
        chroma_where = filters[0]
    else:
        chroma_where = {"$and": filters}

    # ----- semantic query: meaningful leftover tokens -----
    ignored = (
        set(STOPWORDS)
        | set(MONTHS.keys())
        | {"invoice", "receipt", "usd", "eur", "dollar", "dollars",
           "euro", "euros", "over", "greater", "than", "above", "under",
           "less", "below", "paid", "unpaid", "line", "item", "items",
           "outstanding", "overdue", "raw", "text"}
    )
    tokens = tokenize(q)
    semantic_terms = [t for t in tokens if t not in ignored and not t.isdigit() and len(t) > 1]
    semantic_query = " ".join(semantic_terms) or query

    return {
        "intent": "search_documents",
        "semantic_query": semantic_query,
        "chroma_where": chroma_where,
        "source_column": source_column,
        "amount_gt": amount_gt,
        "amount_lt": amount_lt,
    }


# =========================
# 4. POST-FILTER FOR AMOUNTS
# =========================
# ChromaDB stores total/subtotal/tax as strings (per your schema).
# Numeric range filters must be applied after retrieval.

def post_filter_amounts(
    results: dict,
    amount_gt: Optional[float],
    amount_lt: Optional[float],
) -> dict:
    if amount_gt is None and amount_lt is None:
        return results

    kept_ids, kept_docs, kept_metas, kept_dists = [], [], [], []

    for doc_id, doc, meta, dist in zip(
        results["ids"][0],
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        try:
            total = float(meta.get("total", 0))
        except (ValueError, TypeError):
            total = 0.0

        if amount_gt is not None and total <= amount_gt:
            continue
        if amount_lt is not None and total >= amount_lt:
            continue

        kept_ids.append(doc_id)
        kept_docs.append(doc)
        kept_metas.append(meta)
        kept_dists.append(dist)

    return {
        "ids": [kept_ids],
        "documents": [kept_docs],
        "metadatas": [kept_metas],
        "distances": [kept_dists],
    }


# =========================
# 5. RETRIEVAL PIPELINE
# =========================

def hybrid_retrieve(
    user_query: str,
    collection: chromadb.Collection,
    vendor_lookup: dict,           # ← add this
    n_results: int = 5,
) -> dict[str, Any]:
    parsed = parse_user_query(user_query, vendor_lookup)

    print("\n[PARSED QUERY]")
    print(json.dumps(parsed, indent=2, default=str))

    query_kwargs: dict[str, Any] = {
        "query_texts": [parsed["semantic_query"]],
        "n_results": n_results,
        "include": ["documents", "metadatas", "distances"],
    }

    if parsed["chroma_where"]:
        query_kwargs["where"] = parsed["chroma_where"]

    results = collection.query(**query_kwargs)

    # Numeric post-filtering for amount (stored as string)
    results = post_filter_amounts(results, parsed["amount_gt"], parsed["amount_lt"])

    return {
        "parsed": parsed,
        "results": results,
    }


# =========================
# 6. ANSWER FORMATTER
# =========================

def format_answer(user_query: str, output: dict[str, Any]) -> str:
    results = output["results"]
    ids = results["ids"][0]
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0]

    if not ids:
        return "No documents matched your query."

    lines = [f"Query: {user_query}", f"Found {len(ids)} result(s):\n"]

    for rank, (doc_id, doc, meta, dist) in enumerate(zip(ids, docs, metas, dists), 1):
        lines.append(f"  Result #{rank}  (cosine distance: {dist:.4f})")
        lines.append(f"  ID             : {doc_id}")
        lines.append(f"  Source column  : {meta.get('source_column')}")
        lines.append(f"  Vendor         : {meta.get('vendor')}")
        lines.append(f"  Invoice #      : {meta.get('invoice_number')}")
        lines.append(f"  Date           : {meta.get('invoice_date')}")
        lines.append(f"  Total          : {meta.get('total')} {meta.get('currency')}")
        lines.append(f"  Status         : {meta.get('status')}")
        lines.append(f"  Text snippet   : {doc[:200]}...")
        lines.append("")

    return "\n".join(lines)


# =========================
# 7. DEMO
# =========================

if __name__ == "__main__":
    embedding_fn = FinLangEmbeddingFunction(
        model_name="FinLang/finance-embeddings-investopedia",
        device="cpu",
    )

    client = chromadb.PersistentClient(path="./invoices_chroma_db")
    collection = client.get_or_create_collection(
        name="invoices",
        embedding_function=embedding_fn,
        metadata={"hnsw:space": "cosine"},
    )

    print(f"Collection has {collection.count()} vectors.\n")

    # test_queries = [
    #     "Find the Nguyen-Roach invoice from February 2021",
    #     "Show me invoices in USD over 200",
    #     "Search line items for wine glasses",
    #     "Find paid invoices",
    #     "Show invoices in 2021",
    # ]
    
    test_queries = ["Collins Group",]

    for q in test_queries:
        print("=" * 80)
        output = hybrid_retrieve(q, collection, vendor_lookup, n_results=5)
        print(format_answer(q, output))

Loading model 'FinLang/finance-embeddings-investopedia'...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15673.31it/s]


Collection has 998 vectors.


[PARSED QUERY]
{
  "intent": "search_documents",
  "semantic_query": "collins group",
  "chroma_where": {
    "vendor_normalized": {
      "$eq": "collins group"
    }
  },
  "source_column": null,
  "amount_gt": null,
  "amount_lt": null
}
Query: Collins Group
Found 2 result(s):

  Result #1  (cosine distance: 0.7643)
  ID             : 448_raw_text
  Source column  : raw_text
  Vendor         : Collins Group
  Invoice #      : 86225018
  Date           : 2015-12-18
  Total          : 0.0 USD
  Status         : unknown
  Text snippet   : Invoice 86225018 from Collins Group dated 2015-12-18. Items: Fast i5 RGB Gaming Desktop PC Computer nVidia Gefore WiFi Fortnite GTAV PUBG IG3 | FAST Dell Optiplex Windows 10 Desktop Computer Tower C2D...

  Result #2  (cosine distance: 0.9475)
  ID             : 448_line_items
  Source column  : line_items
  Vendor         : Collins Group
  Invoice #      : 86225018
  Date           : 2015-12-18
  Total          : 0.0 USD

In [19]:
output

{'parsed': {'intent': 'search_documents',
  'semantic_query': 'collins group',
  'chroma_where': {'vendor_normalized': {'$eq': 'collins group'}},
  'source_column': None,
  'amount_gt': None,
  'amount_lt': None},
 'results': {'ids': [['448_raw_text', '448_line_items']],
  'embeddings': None,
  'documents': [['Invoice 86225018 from Collins Group dated 2015-12-18. Items: Fast i5 RGB Gaming Desktop PC Computer nVidia Gefore WiFi Fortnite GTAV PUBG IG3 | FAST Dell Optiplex Windows 10 Desktop Computer Tower C2D 4GB DVD WiFi 17" LCD | Dell Desktop Computer PC Optiplex 790 Quad Core i5 8GB 480GB SSD Windows 10 Wifi | Dell Gaming PC 17, NVIDIA GTX 1650, SSD + 1TB, 16GB RAM, WIN10, Desktop Computer | Dell Optiplex 790 Computer i7 @ 3.40 Ghz Quad Core 250GB 4GB Working. Total: None USD.',
    '[{\'description\': \'Fast i5 RGB Gaming Desktop PC Computer nVidia Gefore WiFi Fortnite GTAV PUBG IG3\', \'qty\': 500.0, \'unit_price\': None, \'line_total\': None}, {\'description\': \'FAST Dell Optiplex

In [20]:
r = df[df['File Name']=="batch1-0341.jpg"]

print(r['OCRed Text'].values[0])

Invoice no: 86225018 Date of issue: 12/18/2015 Seller: Client: Collins Group Pope LLC 9443 Poole Knolls Suite 177 98672 Rodriguez Common Apt. 340 Adrianberg, ND 38826 New Isabellabury, NY 68785 Tax Id: 942-76-9532 Tax Id: 918-70-1520 IBAN: GB6OBEERO2797168010013 ITEMS No. Description Qty UM Net price Net worth VAT [%] Gross worth Fast i5 RGB Gaming Desktop PC 5,00 each 510,00 2 550,00 10% 2 805,00 Computer nVidia Gefore WiFi Fortnite GTAV PUBG 1G3 2 FAST Dell Optiplex Windows 10 1,00 each 124,95 124,95 10% 137,45 Desktop Computer Tower CZD 4GB DVD WiFi 17" LCD 3 Dell Desktop Computer PC 5,00 each 144,99 724,95 10% 797,45 Optiplex 790 Quad Core i5 8GB 480GB SSD Windows 10 Wifi Dell Gaming PC I7 , NVIDIA GTX 3,00 each 509,95 1 529,85 10% 1 682,83 1650, SSD + 1TB, 16GB RAM, WIN1O, Desktop Computer 5 Dell Optiplex 790 Computer i7 3,00 each 159,99 479,97 10% 527,97 3.40 Ghz Quad Core 250GB 4GB Working SUMMARY VAT [%] Net worth VAT Gross worth 10% 5 409,72 540,97 5 950,69 Total $ 5 409,72 $ 

# Prompt 

In [21]:
from transformers import pipeline
# from rich import print
import json

classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/ModernBERT-large-zeroshot-v2.0"
)


Loading weights: 100%|██████████| 174/174 [00:00<00:00, 478.43it/s]


In [22]:

text = ["Give invoice for company X between the dates 2026-01-01 and 2026-03-31"]

finance_prompts = [
    "Generate an invoice for company X between 2026-01-01 and 2026-03-31.",
    "Show the total revenue for product Y in Q1 2026.",
    "Send me the bank statement for account 12345 from last month.",
    "Categorize this transaction as travel, office supplies, or salary.",
    "What is the current balance on my credit card ending in 7890?",
    "Calculate the VAT amount for this invoice of 1000 EUR.",
    "Export the list of unpaid invoices to CSV.",
    "Create a budget for the marketing department for 2026.",
    "Tell me how much we spent on cloud hosting in April.",
    "Send a reminder email to client Z about their overdue payment."
]

non_finance_prompts = [
    "Write a short story about a cat in Athens.",
    "Give me ideas for a birthday party for my child.",
    "What are the best restaurants in downtown Athens?",
    "Draft an email to my friend about meeting for coffee.",
    "Explain how photosynthesis works in simple terms."
]

results = classifier(
    non_finance_prompts,
    candidate_labels=["finance", "non-finance"],
    hypothesis_template="This request is about {}."
)

# for result in results:
#     print(result)

results

[{'sequence': 'Write a short story about a cat in Athens.',
  'labels': ['non-finance', 'finance'],
  'scores': [0.9995308518409729, 0.0004691763606388122]},
 {'sequence': 'Give me ideas for a birthday party for my child.',
  'labels': ['non-finance', 'finance'],
  'scores': [0.99993497133255, 6.502816540887579e-05]},
 {'sequence': 'What are the best restaurants in downtown Athens?',
  'labels': ['non-finance', 'finance'],
  'scores': [0.9999822974205017, 1.7778551409719512e-05]},
 {'sequence': 'Draft an email to my friend about meeting for coffee.',
  'labels': ['non-finance', 'finance'],
  'scores': [0.9998601675033569, 0.0001398220774717629]},
 {'sequence': 'Explain how photosynthesis works in simple terms.',
  'labels': ['non-finance', 'finance'],
  'scores': [0.9998623132705688, 0.00013765462790615857]}]

In [ ]:
# =========================
# 8. CLASSIFIER + REFEREE PIPELINE
# =========================
# Optional: validate parser output with an LLM referee
# Only invokes the LLM when the rule-based parser looks uncertain.

import json
import re
from typing import Optional
from google import genai

# ----- 8a. Finance classifier (HF zero-shot) -----
# classifier can be None (skip classification) or a pipeline
# Example init:
#   from transformers import pipeline
#   classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")


genai_client = genai.Client()


# def call_daisy(prompt):
#     response = genai_client.models.generate_content(
#                 model="gemini-3-flash-preview",
#                 contents=prompt,
#             )
#     return response

def call_daisy(prompt):
    # stream = genai_client.models.generate_content_stream(
    # model="gemini-3-flash-preview",
    # contents=prompt,
    # )
    full = ""
    for i, chunk in enumerate(genai_client.models.generate_content_stream(
        model="gemini-3-flash-preview", contents=prompt
    ), 1):
        print(chunk.text, end="", flush=True)
        full += chunk.text
    print()
    return full

def classify_finance(
    queries: list[str],
    classifier: Optional[object] = None,
) -> list[bool]:
    """Return True for each query classified as finance-related."""
    if classifier is None:
        return [True] * len(queries)  # skip classification
    results = classifier(
        queries,
        candidate_labels=["finance", "non-finance"],
        hypothesis_template="This request is about {}.",
    )
    l = [True] * len(queries)
    
    for idx, result in enumerate(results):
        if (result['labels'][0] == 'non-finance') and (result['scores'][0] < 0.59):
            l[idx] = True
        else: l[idx] = True if result['labels'][0] == 'finance' else False
    
    return l


# ----- 8b. Heuristic bypass — skip LLM when parser looks good -----

def needs_referee(user_query: str, parsed: dict) -> bool:
    """Return True if the parser output is uncertain enough to need LLM validation."""
    q = user_query.lower()

    # 1. Parser found nothing structured → likely missed intent
    if parsed.get("chroma_where") is None:
        return True

    # 2. Query hints at money but parser didn't capture it
    has_amount = parsed.get("amount_gt") is not None or parsed.get("amount_lt") is not None
    if ("over" in q or "greater" in q or "above" in q or "$" in q) and not has_amount:
        return True

    # 3. Query mentions a month but parser has no date filter
    months = {"january","february","march","april","may","june",
              "july","august","september","october","november","december"}
    if any(m in q for m in months):
        flat = json.dumps(parsed.get("chroma_where", {}))
        if "$gte" not in flat and "$lt" not in flat:
            return True

    # 4. Looks solid — skip the LLM
    return False


# ----- 8c. Referee LLM prompt (matches current schema) -----

REFEREE_PROMPT = (
    "You are a strict information-extraction referee.\n\n"
    "Your job is to validate whether a parsed finance query matches the original user request.\n"
    "You will receive:\n"
    "1. the original user prompt\n"
    "2. the parser output JSON\n\n"
    "You must compare the parser output against the user prompt and determine whether each extracted field is correct.\n\n"
    "Rules:\n"
    "- Be conservative.\n"
    "- Do not invent values that are not clearly supported by the user prompt.\n"
    "- If a field is missing from the user prompt, it must be null.\n"
    "- If a field in the parser output is wrong, fix it.\n"
    "- Preserve the intended schema exactly.\n"
    "- Return JSON only.\n"
    "- No markdown.\n"
    "- No explanation text outside JSON.\n\n"
    "Expected output schema:\n"
    "{\n"
    '  "is_valid": true,\n'
    '  "errors": [],\n'
    '  "corrected": {\n'
    '    "intent": "search_documents",\n'
    '    "semantic_query": "",\n'
    '    "chroma_where": null,\n'
    '    "source_column": null,\n'
    '    "amount_gt": null,\n'
    '    "amount_lt": null\n'
    "  },\n"
    '  "field_checks": {\n'
    '    "intent": {"ok": true, "reason": ""},\n'
    '    "semantic_query": {"ok": true, "reason": ""},\n'
    '    "chroma_where": {"ok": true, "reason": ""},\n'
    '    "source_column": {"ok": true, "reason": ""},\n'
    '    "amount_gt": {"ok": true, "reason": ""},\n'
    '    "amount_lt": {"ok": true, "reason": ""}\n'
    "  }\n"
    "}\n\n"
    "Validation policy:\n"
    '- "is_valid" is true only if all extracted fields are correct.\n'
    '- "errors" should contain short machine-readable error codes, e.g. ["wrong_doc_type", "wrong_vendor"]\n'
    '- "corrected" must contain the best corrected parse based only on the user prompt.\n'
    '- "chroma_where" uses ChromaDB operators: $eq, $gte, $lt, $in, $and.\n'
    '  Example: {"$and": [{"doc_type": {"$eq": "invoice"}}, {"vendor_normalized": {"$eq": "nguyen-roach"}}]}\n'
    '  Date ranges use invoice_date_int with integer values (YYYYMMDD), like: {"invoice_date_int": {"$gte": 20210201}}\n'
    '- "source_column" should be "line_items", "rawtext", or null (search all).\n'
    '- "amount_gt"/"amount_lt" are numeric thresholds for total amount.\n'
    '- "semantic_query" should contain only leftover meaningful search terms not already represented in chroma_where.\n'
    '- If a value is uncertain, set it to null and mark the relevant field check as not ok.\n\n'
    "Original user prompt:\n"
    "{user_query}\n\n"
    "Parser output JSON:\n"
    "{parser_json}"
)

# ----- 8d. LLM caller (HF or Gemini) -----

def call_referee_llm(
    user_query: str,
    parsed: dict,
    llm_pipe=None,           # HF transformers pipeline
    genai_client=None,       # google.genai Client
    model_name: str = "Qwen/Qwen2.5-3B-Instruct",
) -> dict:
    """Call an LLM to validate/correct the parser output.
    Supports either a HuggingFace text-generation pipeline or a Google GenAI client.
    """
    prompt = REFEREE_PROMPT.format(
        user_query=user_query,
        parser_json=json.dumps(parsed, indent=2, default=str),
    )

    if genai_client is not None:
        response = genai_client.models.generate_content(
            model=model_name,
            contents=prompt,
        )
        text = response.text
    elif llm_pipe is not None:
        messages = [{"role": "user", "content": prompt}]
        out = llm_pipe(messages, max_new_tokens=512, temperature=0.0)
        text = out[0]["generated_text"][-1]["content"]
    else:
        raise ValueError("Provide either llm_pipe or genai_client")

    # Strip markdown fences if present
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[-1]
        text = text.rsplit("```", 1)[0]
    return json.loads(text.strip())


# ----- 8e. End-to-end pipeline -----

def run_pipeline(
    user_query: str,
    collection: object,
    vendor_lookup: dict,
    classifier: Optional[object] = None,
    llm_pipe=None,
    genai_client=None,
    llm_model: str = "Qwen/Qwen2.5-3B-Instruct",
    n_results: int = 5,
) -> dict:
    """Full pipeline: classify → parse → (optional) referee → retrieve."""
    # Step 1: classify
    is_finance = classify_finance([user_query], classifier)[0]
    if not is_finance:
        return {
            "finance_related": False,
            "message": "This query does not appear to be finance-related.",
        }

    # Step 2: parse
    parsed = parse_user_query(user_query, vendor_lookup)

    # Step 3: conditional referee
    referee_result = None
    if needs_referee(user_query, parsed):
        try:
            referee_result = call_referee_llm(
                user_query, parsed,
                llm_pipe=llm_pipe, genai_client=genai_client, model_name=llm_model,
            )
            if referee_result.get("is_valid"):
                parsed = referee_result["corrected"]
        except Exception as e:
            print(f"[Referee LLM failed: {e}] — falling back to rule-based parse")

    # Step 4: retrieve
    output = hybrid_retrieve(user_query, collection, vendor_lookup, n_results=n_results)
    return {
        "finance_related": True,
        "parsed": parsed,
        "referee": referee_result,
        "retrieval": output,
        "answer": format_answer(user_query, output),
    }


In [ ]:

# =========================
# 9. DEMO — full pipeline
# =========================
# Uncomment the classifier / LLM lines below to activate.

# --- Optional: HF classifier ---
# from transformers import pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
# classifier = None  # set to None to skip classification

# --- Optional: HF referee LLM ---
# llm_pipe = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct", device=-1)
llm_pipe = None

# --- Optional: Google GenAI referee ---
# from google import genai
genai_client = genai.Client(api_key="")
# genai_client = None

# --- Run test queries ---
test_queries = [
    # "Find the Nguyen-Roach invoice from February 2021",
    # "Show me invoices in USD over 200",
    # "Search if I bought wine glasses",
    # "Find paid invoices",
    # "Show invoices in 2021",
    # "What is the weather today?",
    # "Does the invoice 1234567 exists?",
    # "What is the capital of Greece?",
    "How much money have I spend on shoes in 2019?"
]



for q in test_queries:
    print("=" * 80)
    result = run_pipeline(
        q, collection, vendor_lookup,
        classifier=classifier,
        llm_pipe=llm_pipe,
        genai_client=genai_client,
        n_results=5,
    )
    if not result["finance_related"]:
        print(f"Query: {q}")
        print(result["message"])
    else:
        if result["referee"]:
            print("[REFEREE VALIDATION]")
            print(json.dumps(result["referee"], indent=2, default=str))
        print(result["answer"])

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 1214.44it/s]



[PARSED QUERY]
{
  "intent": "search_documents",
  "semantic_query": "how much money have spend on shoes",
  "chroma_where": {
    "$and": [
      {
        "invoice_date_int": {
          "$gte": 20190101
        }
      },
      {
        "invoice_date_int": {
          "$lt": 20200101
        }
      }
    ]
  },
  "source_column": null,
  "amount_gt": null,
  "amount_lt": null
}
Query: How much money have I spend on shoes in 2019?
Found 5 result(s):

  Result #1  (cosine distance: 0.5829)
  ID             : 75_raw_text
  Source column  : raw_text
  Vendor         : Gutierrez, Shah and Davis
  Invoice #      : 15203117
  Date           : 2019-05-10
  Total          : 25.52 USD
  Status         : unknown
  Text snippet   : Invoice 15203117 from Gutierrez, Shah and Davis dated 2019-05-10. Items: boys shoes size 5. Total: 25.52 USD....

  Result #2  (cosine distance: 0.6224)
  ID             : 39_raw_text
  Source column  : raw_text
  Vendor         : Reyes, Fox and Martinez
  Invoice

In [25]:
extr_docs_filenames = []
for doc in result['retrieval']['results']['metadatas'][0]:
    extr_docs_filenames.append(doc['filename'])


extr_docs_filenames

['batch1-0169.jpg',
 'batch1-0391.jpg',
 'batch1-0406.jpg',
 'batch1-0288.jpg',
 'batch1-0218.jpg']

In [26]:
retrieved_documents = df[df['File Name'].apply(lambda x: x in extr_docs_filenames)]['OCRed Text'].tolist()

retrieved_documents

['Invoice no: 84423800 Date of issue: 10/11/2019 Seller: Client: Reyes, Fox and Martinez Morgan Ltd 8258 Lang Trail Apt: 792 316 Alexander Run Apt: 930 Port Jasonhaven, NJ 07777 South Danabury, WV 35900 Tax Id: 913-91-4093 Tax Id: 945-98-2210 IBAN: GBO6YQKI12270767261206 ITEMS No. Description Qty UM Net price Net worth VAT [%] Gross worth PUMA RS DREAMER SUPER 2,00 each 125,00 250,00 10% 275,00 MARIO 64 NINTENDO Little Kids US 2.5 2 kid shoes 4,00 each 50,00 200,00 10% 220,00 3 Newborn Infant Baby Girl Boy 5,00 each 22,69 113,45 10% 124,80 Kid Winter Warm Coat Knit Outwear Hooded Jumpsuit 5 YOUTH Boys Big Kids Nike 3,00 each 89,99 269,97 10% 296,97 Jordan 6-17-23 Basketball White red Black 428818 100 5 PUMA Youth Universal FG 5,00 each 19,99 99,95 10% 109,94 Jr Soccer Cleats 5 White High Risk Red Black 5Y SUMMARY VAT [%] Net worth VAT Gross worth 10% 933,37 93,34 1 026,71 Total $ 933,37 $ 93,34 $ 1 026,71 Boys',
 'Invoice no: 15203117 Date of issue: 05/10/2019 Seller: Client: Gutierrez

In [27]:
ANSWER_PROMPT = (
    "You are a financial document analyst. Answer the user's question using ONLY the provided documents.\n\n"
    "Rules:\n"
    "- Base your answer strictly on the documents below.\n"
    "- If the documents do not contain the information requested, say so.\n"
    "- Cite specific invoice numbers, vendors, dates, and amounts when relevant.\n"
    "- Be concise and direct.\n\n"
    "User query:\n"
    f"{test_queries}\n\n"
    "Retrieved documents:\n"
    f"{retrieved_documents}\n\n"
    "Answer:"
)

In [28]:
print(ANSWER_PROMPT)

You are a financial document analyst. Answer the user's question using ONLY the provided documents.

Rules:
- Base your answer strictly on the documents below.
- If the documents do not contain the information requested, say so.
- Cite specific invoice numbers, vendors, dates, and amounts when relevant.
- Be concise and direct.

User query:
['How much money have I spend on shoes in 2019?']

Retrieved documents:
['Invoice no: 84423800 Date of issue: 10/11/2019 Seller: Client: Reyes, Fox and Martinez Morgan Ltd 8258 Lang Trail Apt: 792 316 Alexander Run Apt: 930 Port Jasonhaven, NJ 07777 South Danabury, WV 35900 Tax Id: 913-91-4093 Tax Id: 945-98-2210 IBAN: GBO6YQKI12270767261206 ITEMS No. Description Qty UM Net price Net worth VAT [%] Gross worth PUMA RS DREAMER SUPER 2,00 each 125,00 250,00 10% 275,00 MARIO 64 NINTENDO Little Kids US 2.5 2 kid shoes 4,00 each 50,00 200,00 10% 220,00 3 Newborn Infant Baby Girl Boy 5,00 each 22,69 113,45 10% 124,80 Kid Winter Warm Coat Knit Outwear Hoode

In [30]:
res = call_daisy(ANSWER_PROMPT)

Based on the provided documents, you spent a total of **$928.52** on shoes in 2019.

The spending is detailed across the following invoices:

*   **Invoice 84423800** (Date: 10/11/2019, Seller: Reyes, Fox and Martinez): **$901.91**
    *   PUMA RS DREAMER SUPER MARIO 64: $275.00
    *   kid shoes: $220.00
    *   YOUTH Boys Nike Jordan Basketball shoes: $296.97
    *   PUMA Youth Soccer Cleats: $109.94
*   **Invoice 15203117** (Date: 05/10/2019, Seller: Gutierrez, Shah and Davis): **$25.52**
    *   boys shoes size 5: $25.52
*   **Invoice 64855498** (Date: 08/01/2019, Seller: Johnston-Douglas): **$1.09**
    *   Lakai Select Skate Shoe: $1.09


In [ ]:
# the memory should contain the files rettrieved so far

# MEMORY